# 01 — Understanding the CAMUS Dataset

## Learning objectives

By the end of this notebook, I should be able to:

1. Explain how patients, cardiac views, cardiac phases, images, and masks are related.
2. Load one CAMUS ultrasound image and its corresponding segmentation mask.
3. Inspect image shape, data type, pixel spacing, intensity range, and mask labels.
4. Explain what one training sample represents in a semantic segmentation task.
5. Identify data-quality and data-splitting checks that prevent invalid experiments.

## Working method

Each section follows the same cycle:

1. State the question that the code should answer.
2. Write and run the smallest useful code cell.
3. Interpret the output in plain language.
4. Record a conclusion before moving to the next section.

## Step 1 — Verify the notebook environment

Before reading data, verify the notebook's current working directory and confirm that NiBabel is available in the selected kernel. Relative file paths are resolved from the current working directory, so an incorrect directory can make valid project paths appear to be missing.

**Coding task:** Create a code cell below that:

- imports `Path` from `pathlib`;
- imports `nibabel` using the alias `nib`;
- prints the current working directory;
- prints the installed NiBabel version.


In [ ]:
from pathlib import Path
import nibabel as nib

print("Current working directory:", Path.cwd())
print("Nibabel version:", nib.__version__)


### Step 1 conclusion

The notebook kernel is using an environment that provides NiBabel 5.4.2. Its current working directory is the project's `notebooks/` directory, not the project root. Therefore, paths such as `data/raw/camus` would be resolved from the wrong location if they were used directly.

## Step 2 — Define explicit project and data paths

The project root is the parent of the current `notebooks/` directory. Building paths from that known relationship is clearer and more reproducible than hard-coding an absolute path tied to one computer.

`pathlib.Path` uses the `/` operator to join path components. In this context, `/` constructs a new path; it is not numeric division.

Use uppercase names for these values because they act as notebook-level configuration constants.

**Coding task:** Create a code cell below that:

1. stores the current working directory in `NOTEBOOK_DIR`;
2. derives `PROJECT_ROOT` from the parent of `NOTEBOOK_DIR`;
3. constructs `DATA_DIR` for the project's top-level `data/` directory;
4. constructs `CAMUS_ROOT` for `data/raw/camus`;
5. prints all four paths;
6. prints whether `CAMUS_ROOT` is an existing directory.


In [ ]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"
CAMUS_ROOT = DATA_DIR / "raw" / "camus"

print("Notebook directory:", NOTEBOOK_DIR)
print("Project root directory:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("CAMUS directory:", CAMUS_ROOT)
print("CAMUS directory exists:", CAMUS_ROOT.is_dir())

### Step 2 conclusion

The notebook now derives every project path from its known working directory. `CAMUS_ROOT` points to the intended `data/raw/camus` directory, and `is_dir()` confirms that the directory exists. This avoids machine-specific absolute paths while still failing clearly if the project structure changes.

## Step 3 — Identify and count patient directories

CAMUS is organized primarily by patient. Each directory named `patientXXXX` represents one patient and contains multiple views, phases, images, masks, and metadata files. A patient is therefore the independent subject unit; an individual image is not an independent patient.

`Path.glob("patient*")` finds entries whose names begin with `patient`. Filtering with `is_dir()` prevents a similarly named file from being counted as a patient. `sorted()` gives a deterministic order, which makes inspection and later data processing reproducible.

**Coding task:** Create a code cell below that:

1. finds entries under `CAMUS_ROOT` whose names begin with `patient`;
2. keeps only directories;
3. sorts the resulting paths and stores them in `patient_dirs`;
4. prints the number of patient directories;
5. prints the names of the first three and last three patients.

The official CAMUS dataset contains 500 patients. The purpose of this check is to verify the local dataset rather than simply assume that the download is complete.


In [4]:
patient_dirs = sorted(
    path
    for path in CAMUS_ROOT.glob("patient*")
    if path.is_dir()
)

print("Number of patient directories:", len(patient_dirs))
print("First three patients:", [path.name for path in patient_dirs[:3]])
print("Last three patients:", [path.name for path in patient_dirs[-3:]])

Number of patient directories: 500
First three patients: ['patient0001', 'patient0002', 'patient0003']
Last three patients: ['patient0498', 'patient0499', 'patient0500']


### Step 3 conclusion

The local dataset contains 500 patient directories, from `patient0001` through `patient0500`. This matches the expected CAMUS patient count. It verifies the top-level patient inventory, but it does not yet prove that every patient directory contains the required files.

## Step 4 — Inspect one patient directory

Before writing a dataset-wide validator, inspect one representative patient. This reveals the naming convention and the relationship between metadata, cardiac views, cardiac phases, ultrasound images, and segmentation masks.

`patient_dirs[0]` selects the first path from the sorted patient list. `Path.iterdir()` iterates over the direct children of that directory. Filtering with `is_file()` excludes any nested directories, and sorting makes the displayed order deterministic.

The purpose of this step is observation: list the files first, then interpret their names. Do not build filename assumptions into a data loader before verifying the actual data layout.


In [5]:
sample_patient_dir = patient_dirs[0]
sample_files = sorted(
    path
    for path in sample_patient_dir.iterdir()
    if path.is_file()
)

print("Selected patient:", sample_patient_dir.name)
print("Number of files:", len(sample_files))
print("Files:")

for path in sample_files:
    print(" -", path.name)

Selected patient: patient0001
Number of files: 15
Files:
 - Info_2CH.cfg
 - Info_4CH.cfg
 - MANDATORY_CITATION.md
 - patient0001_2CH_ED.nii.gz
 - patient0001_2CH_ED_gt.nii.gz
 - patient0001_2CH_ES.nii.gz
 - patient0001_2CH_ES_gt.nii.gz
 - patient0001_2CH_half_sequence.nii.gz
 - patient0001_2CH_half_sequence_gt.nii.gz
 - patient0001_4CH_ED.nii.gz
 - patient0001_4CH_ED_gt.nii.gz
 - patient0001_4CH_ES.nii.gz
 - patient0001_4CH_ES_gt.nii.gz
 - patient0001_4CH_half_sequence.nii.gz
 - patient0001_4CH_half_sequence_gt.nii.gz


### Step 4 conclusion

`patient0001` contains 15 files:

- two view-specific configuration files;
- one mandatory citation notice;
- twelve NIfTI files: two views × three data variants × two roles.

The filename components have the following meanings:

| Component | Meaning |
|---|---|
| `2CH` | Apical two-chamber view |
| `4CH` | Apical four-chamber view |
| `ED` | End-diastole, when the left ventricle is typically at its largest |
| `ES` | End-systole, when the left ventricle is typically at its smallest |
| `_gt` | Ground-truth segmentation annotation |
| `half_sequence` | A multi-frame sequence variant; its exact dimensions and label content must be verified from the data |
| `.nii.gz` | A gzip-compressed NIfTI medical-image file |

The `MANDATORY_CITATION.md` file is part of the dataset usage requirements, not a training sample. The final repository and project report must preserve the required CAMUS citation.

## Step 5 — Read the view metadata as raw text

`Info_2CH.cfg` and `Info_4CH.cfg` contain metadata for the two cardiac views. Read the original text before writing a parser so that the parser is based on observed syntax rather than assumptions.

`Path.read_text()` reads a text file into a Python string. Printing the two files separately makes differences between views visible.


In [6]:
info_2ch_path = sample_patient_dir / "Info_2CH.cfg"
info_4ch_path = sample_patient_dir / "Info_4CH.cfg"

print("2CH metadata:")
print(info_2ch_path.read_text())

print("4CH metadata:")
print(info_4ch_path.read_text())

2CH metadata:
ED: 1
ES: 18
NbFrame: 18
Sex: F
Age: 56
ImageQuality: Good
EF: 54
FrameRate: 48.4

4CH metadata:
ED: 1
ES: 20
NbFrame: 20
Sex: F
Age: 56
ImageQuality: Good
EF: 54
FrameRate: 48.4



### Step 5 conclusion

The configuration files use one `key: value` pair per line. For `patient0001`, the 2CH and 4CH acquisitions have different end-systolic frame numbers and different frame counts. View-specific frame indices must therefore be read from the corresponding configuration file rather than copied between views.

| Field | Meaning | Example |
|---|---|---|
| `ED` | End-diastolic frame number | `1` |
| `ES` | End-systolic frame number | `18` or `20` |
| `NbFrame` | Number of frames in the view sequence | `18` or `20` |
| `Sex` | Recorded patient sex | `F` |
| `Age` | Patient age in years | `56` |
| `ImageQuality` | Categorical image-quality assessment | `Good` |
| `EF` | Left-ventricular ejection fraction, expressed as a percentage | `54` |
| `FrameRate` | Acquisition rate in frames per second | `48.4` |

The frame numbers in the configuration appear in human-oriented numbering. Python arrays use zero-based indices, but no conversion should be made until the sequence dimensions have been inspected and the numbering convention has been verified.

## Step 6 — Parse configuration text into a dictionary

A Python dictionary stores key-value pairs. Parsing the file into a dictionary allows code to request a field by name, such as `config["ED"]`, instead of depending on line positions.

This first parser deliberately preserves every value as text. Converting fields to integers or floating-point numbers is a separate validation step. Keeping those operations separate makes incorrect assumptions easier to detect.


In [7]:
def read_config(path):
    config = {}

    for line in path.read_text().splitlines():
        if not line.strip():
            continue

        key, value = line.split(":", maxsplit=1)
        config[key.strip()] = value.strip()

    return config


config_2ch = read_config(info_2ch_path)
config_4ch = read_config(info_4ch_path)

print("Parsed 2CH configuration:", config_2ch)
print("Parsed 4CH configuration:", config_4ch)
print("Stored type of 2CH ED:", type(config_2ch["ED"]).__name__)

Parsed 2CH configuration: {'ED': '1', 'ES': '18', 'NbFrame': '18', 'Sex': 'F', 'Age': '56', 'ImageQuality': 'Good', 'EF': '54', 'FrameRate': '48.4'}
Parsed 4CH configuration: {'ED': '1', 'ES': '20', 'NbFrame': '20', 'Sex': 'F', 'Age': '56', 'ImageQuality': 'Good', 'EF': '54', 'FrameRate': '48.4'}
Stored type of 2CH ED: str


### Step 6 conclusion

The configuration text has been parsed into dictionaries, but every value is still a string. Field names provide structure; field types provide usable meaning. Numeric validation must happen after explicit conversion.

## Step 7 — Convert metadata to semantic types and validate frame ranges

`ED`, `ES`, `NbFrame`, `Age`, and `EF` are stored as integers. `FrameRate` is stored as a floating-point number. `Sex` and `ImageQuality` remain strings. The conversion function below applies these rules consistently to both views.

The frame checks verify that ED and ES fall within `1..NbFrame`. This is consistent with one-based frame numbering in the configuration; conversion to zero-based NumPy indices will only be needed when indexing a sequence array.


In [9]:
INTEGER_FIELDS = {"ED", "ES", "NbFrame", "Age", "EF"}
FLOAT_FIELDS = {"FrameRate"}


def convert_config_types(config):
    typed_config = {}

    for key, value in config.items():
        if key in INTEGER_FIELDS:
            typed_config[key] = int(value)
        elif key in FLOAT_FIELDS:
            typed_config[key] = float(value)
        else:
            typed_config[key] = value

    return typed_config


typed_config_2ch = convert_config_types(config_2ch)
typed_config_4ch = convert_config_types(config_4ch)

for view_name, config in [("2CH", typed_config_2ch), ("4CH", typed_config_4ch)]:
    assert 1 <= config["ED"] <= config["NbFrame"], f"{view_name}: ED is out of range"
    assert 1 <= config["ES"] <= config["NbFrame"], f"{view_name}: ES is out of range"

print("Typed 2CH configuration:", typed_config_2ch)
print("2CH ED type:", type(typed_config_2ch["ED"]).__name__)
print("2CH FrameRate type:", type(typed_config_2ch["FrameRate"]).__name__)
print("Frame-range validation: passed")

Typed 2CH configuration: {'ED': 1, 'ES': 18, 'NbFrame': 18, 'Sex': 'F', 'Age': 56, 'ImageQuality': 'Good', 'EF': 54, 'FrameRate': 48.4}
2CH ED type: int
2CH FrameRate type: float
Frame-range validation: passed


## Step 8 — Load one image-mask pair

One supervised segmentation sample consists of an input ultrasound image and its spatially aligned ground-truth mask. Here the sample is the 2CH end-diastolic frame from `patient0001`.

`nib.load()` creates a NIfTI image object containing a header, an affine transform, and a lazy data proxy. `np.asarray(nifti_image.dataobj)` materializes the stored array while preserving its represented data type, unlike `get_fdata()`, which defaults to `float64`.

A valid image-mask pair should have matching shapes and matching affine transforms. Shape alignment matches array locations; affine alignment matches their physical coordinate systems.


In [8]:
import numpy as np

patient_id = sample_patient_dir.name
image_path = sample_patient_dir / f"{patient_id}_2CH_ED.nii.gz"
mask_path = sample_patient_dir / f"{patient_id}_2CH_ED_gt.nii.gz"

image_nii = nib.load(image_path)
mask_nii = nib.load(mask_path)
image = np.asarray(image_nii.dataobj)
mask = np.asarray(mask_nii.dataobj)

print("Image path:", image_path.name)
print("Mask path:", mask_path.name)
print("Image shape / dtype:", image.shape, image.dtype)
print("Mask shape / dtype:", mask.shape, mask.dtype)
print("Pixel spacing:", image_nii.header.get_zooms())
print("Spatial units:", image_nii.header.get_xyzt_units()[0])
print("Image intensity range:", float(image.min()), float(image.max()))
print("Mask labels:", np.unique(mask).tolist())
print("Shapes match:", image.shape == mask.shape)
print("Affines match:", np.allclose(image_nii.affine, mask_nii.affine))

Image path: patient0001_2CH_ED.nii.gz
Mask path: patient0001_2CH_ED_gt.nii.gz
Image shape / dtype: (549, 389) float32
Mask shape / dtype: (549, 389) float32
Pixel spacing: (np.float32(0.308), np.float32(0.308))
Spatial units: mm
Image intensity range: 0.0 255.0
Mask labels: [0.0, 1.0, 2.0, 3.0]
Shapes match: True
Affines match: True
